# Constraints

Everything that decides which solutions are *allowed*: time windows, capacities
(one- and two-dimensional), route limits, driver shifts, pickup & delivery,
multi-trip reloads, heterogeneous fleets and multiple depots.

Each section is self-contained — run the setup cell, then jump anywhere.
Docs: Features section of the documentation site.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt

from tqrouting import TQRouter, VRPInstance, Customer, VehicleType, Location, TimeWindow
from tqrouting.matrix import EuclideanMatrixProvider

data_dir = Path("data")

# Uses the TQROUTING_LICENSE_KEY environment variable by default.
# Set it before running: export TQROUTING_LICENSE_KEY="key/..."
router = TQRouter()

## 1. Time windows

Customers accept service only inside `[start, end]`. Times are seconds or
`"HH:MM[:SS]"` strings. The solver may arrive early and wait, or you can make
windows soft (see [objectives.ipynb](objectives.ipynb), soft constraints).

Here we solve `vrptw_100.json`: 100 customers in ten clusters, tight windows.

In [2]:
instance_tw = VRPInstance.from_json(data_dir / "vrptw_100.json")
print(f"VRPTW: {len(instance_tw.customers)} customers")

solution_tw = router.solve(instance_tw, time_limit=5)
print(f"Status: {solution_tw.solution_status}, routes: {len(solution_tw.routes)}")

route = solution_tw.routes[0]
print(f"\nRoute {route.id} arrival times vs windows:")
for visit in route.customer_sequence[:5]:
    c = instance_tw.customers[visit.customer_id]
    print(f"  customer {visit.customer_id}: ETA {visit.eta:.0f}  "
          f"window [{c.time_window.start:.0f}, {c.time_window.end:.0f}]")

[tqrouting] Solving VRP instance (100 customers, 1 vehicle types)...


VRPTW: 100 customers


[tqrouting] Solver finished in 5.0s (status=Feasible, 12 routes, duration 11133.9, 0 unvisited).


Status: Feasible, routes: 12

Route 0 arrival times vs windows:
  customer 10: ETA 58  window [22, 104]
  customer 11: ETA 150  window [112, 156]
  customer 12: ETA 244  window [202, 244]
  customer 14: ETA 340  window [364, 417]
  customer 16: ETA 458  window [560, 645]


## 2. Capacity — one dimension

`capacity_load` on the vehicle vs `delivery_load` / `pickup_load` on customers.
Total demand 36 against capacity 20 forces at least two routes.

In [3]:
instance_cap = VRPInstance(
    customers=[
        Customer(location=Location(x=10, y=0), delivery_load=9),
        Customer(location=Location(x=20, y=10), delivery_load=9),
        Customer(location=Location(x=10, y=20), delivery_load=9),
        Customer(location=Location(x=0, y=10), delivery_load=9),
    ],
    fleet=[VehicleType(start_location=Location(x=10, y=10), capacity_load=20, n_vehicles=3)],
)
instance_cap.compute_duration_matrix(EuclideanMatrixProvider())
sol_cap = router.solve(instance_cap, time_limit=2)
for r in sol_cap.routes:
    print(f"Route {r.id}: load {r.route_delivery_load:.0f} / 20")

[tqrouting] Computing duration matrix for 5 unique locations...


[tqrouting] Duration matrix ready (5×5, 0.0s).


[tqrouting] Solving VRP instance (4 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 3 routes, duration 44.1, 0 unvisited).


Route 0: load 18 / 20
Route 1: load 9 / 20
Route 2: load 9 / 20


## 3. Capacity — weight and volume together (2-D)

Declare both `capacity_load` (weight) and `capacity_volume` and give customers
demands in both dimensions. A vehicle must fit **both** at once.

Classic trap: with vehicles `[10 kg, 1 m3]` and `[1 kg, 10 m3]`, a customer
needing `[8, 8]` fits neither — validation rejects it before the solver runs.

In [4]:
from tqrouting.exceptions import ValidationError

bad = VRPInstance(
    customers=[Customer(location=Location(x=5, y=5), delivery_load=8, delivery_volume=8)],
    fleet=[
        VehicleType(start_location=Location(x=0, y=0), capacity_load=10, capacity_volume=1),
        VehicleType(start_location=Location(x=0, y=0), capacity_load=1, capacity_volume=10),
    ],
)
bad.compute_duration_matrix(EuclideanMatrixProvider())
try:
    router.solve(bad, time_limit=1)
except ValidationError as e:
    print(f"Rejected before solving:\n  {e}")

[tqrouting] Computing duration matrix for 2 unique locations...


[tqrouting] Duration matrix ready (2×2, 0.0s).


Rejected before solving:
  Customer 0 peak load (weight=8, volume=8) cannot be carried by any single vehicle type: no type satisfies both dimensions at once


In [5]:
# Add a vehicle type that dominates both dimensions and it solves fine.
ok = VRPInstance(
    customers=[
        Customer(location=Location(x=5, y=5), delivery_load=8, delivery_volume=8),
        Customer(location=Location(x=8, y=2), delivery_load=2, delivery_volume=9),
    ],
    fleet=[
        VehicleType(id="small", start_location=Location(x=0, y=0), capacity_load=10, capacity_volume=1, n_vehicles=2),
        VehicleType(id="box", start_location=Location(x=0, y=0), capacity_load=10, capacity_volume=10, n_vehicles=2),
    ],
)
ok.compute_duration_matrix(EuclideanMatrixProvider())
sol_2d = router.solve(ok, time_limit=2)
for r in sol_2d.routes:
    print(f"Route {r.id} on vehicle type '{r.vehicle_type_id}': "
          f"load {r.route_delivery_load}")

[tqrouting] Computing duration matrix for 3 unique locations...


[tqrouting] Duration matrix ready (3×3, 0.0s).


[tqrouting] Solving VRP instance (2 customers, 2 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 2 routes, duration 15.3, 0 unvisited).


Route 0 on vehicle type 'box': load [8.0, 8.0]
Route 1 on vehicle type 'box': load [2.0, 9.0]


### Different rules per dimension

The two dimensions are fully independent: each has its own penalty
(`penalty_capacity_load` / `penalty_capacity_volume`), and either can be soft
while the other stays hard. Here a parcel physically cannot fit (volume 5
against capacity 4) — with hard limits validation rejects the instance
outright, but declaring volume soft prices the overflow instead:

In [6]:
overflow = VRPInstance(
    customers=[Customer(location=Location(x=5, y=5), delivery_load=6, delivery_volume=5)],
    fleet=[VehicleType(start_location=Location(x=0, y=0), capacity_load=10, capacity_volume=4)],
)
overflow.compute_duration_matrix(EuclideanMatrixProvider())

try:
    router.solve(overflow, time_limit=2)
except ValidationError as e:
    print(f"Both dimensions hard -> rejected:\n  {e}")

sol_soft = router.solve(
    overflow,
    soft_constraints=["capacity_volume"],   # weight stays hard
    penalty_capacity_volume=200.0,          # price per unit of volume overflow
    time_limit=2,
)
print(f"\nVolume soft: {sol_soft.solution_status}")
for v in sol_soft.constraint_violations:
    print(f"  {v.constraint} (dimension {v.dimension}): excess {v.excess_value}")

[tqrouting] Computing duration matrix for 2 unique locations...


[tqrouting] Duration matrix ready (2×2, 0.0s).


[tqrouting] Solving VRP instance (1 customers, 1 vehicle types)...


Both dimensions hard -> rejected:
  Customer 0 peak load (weight=6, volume=5) cannot be carried by any single vehicle type: no type satisfies both dimensions at once


[tqrouting] Solver finished in 0.4s (status=FeasibleWithSoftViolations, 1 routes, duration 7.1, 0 unvisited).



Volume soft: FeasibleWithSoftViolations
  capacity (dimension 1): excess 1.0


## 4. Route limits — max_duration / max_distance

`max_duration` caps each route's total time (driving + service + waiting).
A 1-hour cap over ~2 hours of total work forces a second vehicle.
(`max_distance` works the same way and needs a separate distance matrix —
see [real_world_osrm.ipynb](real_world_osrm.ipynb).)

In [7]:
instance_lim = VRPInstance(
    customers=[
        Customer(location=Location(x=0, y=1800), service_time=600),
        Customer(location=Location(x=1800, y=0), service_time=600),
        Customer(location=Location(x=0, y=-1800), service_time=600),
    ],
    fleet=[
        VehicleType(
            start_location=Location(x=0, y=0),
            max_duration="01:00:00",
            n_vehicles=3,
        ),
    ],
)
instance_lim.compute_duration_matrix(EuclideanMatrixProvider())
sol_lim = router.solve(instance_lim, time_limit=2)
for r in sol_lim.routes:
    print(f"Route {r.id}: duration {r.route_duration:.0f}s (cap 3600s)")

[tqrouting] Computing duration matrix for 4 unique locations...


[tqrouting] Duration matrix ready (4×4, 0.0s).


[tqrouting] Solving VRP instance (3 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 3 routes, duration 7200.0, 0 unvisited).


Route 0: duration 2400s (cap 3600s)
Route 1: duration 2400s (cap 3600s)
Route 2: duration 2400s (cap 3600s)


## 5. Driver shifts — shift_start / shift_end

A vehicle only operates inside its shift. Two types with staggered shifts:
morning customers land on the early truck, evening ones on the late truck.

In [8]:
instance_shift = VRPInstance(
    customers=[
        Customer(id="morning", location=Location(x=600, y=0),
                 time_window=TimeWindow(start="08:00", end="10:00")),
        Customer(id="evening", location=Location(x=0, y=600),
                 time_window=TimeWindow(start="17:00", end="19:00")),
    ],
    fleet=[
        VehicleType(id="early", start_location=Location(x=0, y=0),
                    shift_start="07:00", shift_end="12:00"),
        VehicleType(id="late", start_location=Location(x=0, y=0),
                    shift_start="15:00", shift_end="21:00"),
    ],
)
instance_shift.compute_duration_matrix(EuclideanMatrixProvider())
sol_shift = router.solve(instance_shift, time_limit=2)
for r in sol_shift.routes:
    stops = [v.customer_id for v in r.customer_sequence]
    print(f"Vehicle '{r.vehicle_type_id}': {stops}, departs {r.route_start_time:.0f}s")

[tqrouting] Computing duration matrix for 3 unique locations...


[tqrouting] Duration matrix ready (3×3, 0.0s).


[tqrouting] Solving VRP instance (2 customers, 2 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 2 routes, duration 1200.0, 0 unvisited).


Vehicle 'early': ['morning'], departs 28200s
Vehicle 'late': ['evening'], departs 60600s


## 6. Pickup & delivery

`delivery_load` leaves the depot on the vehicle; `pickup_load` returns to it.
The on-board peak per customer is `max(delivery, pickup)` — deliveries are
dropped before pickups are taken on board.

In [9]:
instance_pd = VRPInstance(
    customers=[
        Customer(id="drop", location=Location(x=10, y=0), delivery_load=12),
        Customer(id="take", location=Location(x=20, y=10), pickup_load=10),
        Customer(id="both", location=Location(x=10, y=20), delivery_load=6, pickup_load=8),
    ],
    fleet=[VehicleType(start_location=Location(x=0, y=0), capacity_load=15, n_vehicles=2)],
)
instance_pd.compute_duration_matrix(EuclideanMatrixProvider())
sol_pd = router.solve(instance_pd, time_limit=2)
for r in sol_pd.routes:
    print(f"Route {r.id}: delivery {r.route_delivery_load}, "
          f"customers {[v.customer_id for v in r.customer_sequence]}")

[tqrouting] Computing duration matrix for 4 unique locations...


[tqrouting] Duration matrix ready (4×4, 0.0s).


[tqrouting] Solving VRP instance (3 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 2 routes, duration 46.5, 0 unvisited).


Route 0: delivery 6.0, customers ['both']
Route 1: delivery 12.0, customers ['drop', 'take']


## 7. Multi-trip routes — reload

With `reload`, a vehicle can return to its start depot mid-route, refill
(taking `reload` time), and continue — one physical vehicle, several trips.
Six customers of 4 units each (24 total) against capacity 10:

In [10]:
def reload_customers():
    return [
        Customer(location=Location(x=300, y=0), delivery_load=4),
        Customer(location=Location(x=300, y=300), delivery_load=4),
        Customer(location=Location(x=0, y=300), delivery_load=4),
        Customer(location=Location(x=-300, y=300), delivery_load=4),
        Customer(location=Location(x=-300, y=0), delivery_load=4),
        Customer(location=Location(x=-300, y=-300), delivery_load=4),
    ]

# Without reload: 24 units / capacity 10 -> needs 3 vehicles.
no_reload = VRPInstance(
    customers=reload_customers(),
    fleet=[VehicleType(start_location=Location(x=0, y=0), capacity_load=10, n_vehicles=3)],
)
no_reload.compute_duration_matrix(EuclideanMatrixProvider())
sol_no = router.solve(no_reload, time_limit=2)
print(f"Without reload: {len(sol_no.routes)} vehicles")

# With reload: a single vehicle does it in several trips.
with_reload = VRPInstance(
    customers=reload_customers(),
    fleet=[VehicleType(start_location=Location(x=0, y=0), capacity_load=10,
                       n_vehicles=1, reload="00:05:00")],
)
with_reload.compute_duration_matrix(EuclideanMatrixProvider())
sol_re = router.solve(with_reload, time_limit=2)
route = sol_re.routes[0]
print(f"With reload: {len(sol_re.routes)} vehicle, "
      f"{len(route.reload_stops)} reload stop(s)")
for item in route.customer_sequence:
    kind = "RELOAD" if item in route.reload_stops else f"customer {item.customer_id}"
    print(f"  {kind} @ {item.eta:.0f}s")

[tqrouting] Computing duration matrix for 7 unique locations...


[tqrouting] Duration matrix ready (7×7, 0.0s).


[tqrouting] Solving VRP instance (6 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 3 routes, duration 1800.0, 0 unvisited).


[tqrouting] Computing duration matrix for 7 unique locations...


[tqrouting] Duration matrix ready (7×7, 0.0s).


[tqrouting] Solving VRP instance (6 customers, 1 vehicle types)...


Without reload: 3 vehicles


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 3248.5, 0 unvisited).


With reload: 1 vehicle, 2 reload stop(s)
  customer 3 @ 424s
  customer 2 @ 724s
  RELOAD @ 1024s
  customer 0 @ 1624s
  customer 1 @ 1924s
  RELOAD @ 2349s
  customer 4 @ 2949s
  customer 5 @ 3249s


## 8. Heterogeneous fleet

Different vehicle types with different capacities, costs, shifts, or limits.
The solver picks which type serves which customers.

In [11]:
instance_fleet = VRPInstance(
    customers=[
        Customer(location=Location(x=10, y=10), delivery_load=18),
        Customer(location=Location(x=20, y=5), delivery_load=3),
        Customer(location=Location(x=5, y=20), delivery_load=4),
        Customer(location=Location(x=25, y=15), delivery_load=16),
    ],
    fleet=[
        VehicleType(id="van", start_location=Location(x=0, y=0), capacity_load=8, n_vehicles=2),
        VehicleType(id="lorry", start_location=Location(x=0, y=0), capacity_load=20, n_vehicles=2),
    ],
)
instance_fleet.compute_duration_matrix(EuclideanMatrixProvider())
sol_fleet = router.solve(instance_fleet, time_limit=2)
for r in sol_fleet.routes:
    print(f"'{r.vehicle_type_id}' route: load {r.route_delivery_load:.0f}, "
          f"stops {len(r.customer_sequence)}")

[tqrouting] Computing duration matrix for 5 unique locations...


[tqrouting] Duration matrix ready (5×5, 0.0s).


[tqrouting] Solving VRP instance (4 customers, 2 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 3 routes, duration 66.6, 0 unvisited).


'van' route: load 4, stops 1
'lorry' route: load 18, stops 1
'lorry' route: load 19, stops 2


## 9. Multiple depots

Vehicle types can start from different locations — `mdvrp_100.json` has two
depots with five vehicles each. Each route starts and ends at its own vehicle's depot.

In [12]:
instance_md = VRPInstance.from_json(data_dir / "mdvrp_100.json")
print(f"MDVRP: {len(instance_md.customers)} customers, {len(instance_md.fleet)} depots/types")
sol_md = router.solve(instance_md, time_limit=5)
print(f"Status: {sol_md.solution_status}, routes: {len(sol_md.routes)}")
per_type = {}
for r in sol_md.routes:
    per_type[r.vehicle_type_id] = per_type.get(r.vehicle_type_id, 0) + 1
print("Routes per depot/type:", per_type)

[tqrouting] Solving VRP instance (100 customers, 2 vehicle types)...


MDVRP: 100 customers, 2 depots/types


[tqrouting] Solver finished in 5.0s (status=Feasible, 9 routes, duration 857.5, 0 unvisited).


Status: Feasible, routes: 9
Routes per depot/type: {0: 4, 1: 5}
